# Agentic AI: Langchain 

# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` ou ` uv add langchain-ollama ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [5]:
from langchain_ollama import ChatOllama
from IPython.display import display, Markdown
from langchain.agents import create_agent

print("Imports réussis !")

Imports réussis !


In [6]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

Agent Simple

In [7]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="gemma4:31b-cloud", temperature=0.1)

agent=create_agent(
  model=llm,
  tools=[],
  system_prompt=" You are a helpful assistant"
)

In [8]:
response=agent.invoke(input={
        'messages':[
            {"role":"user", "content":"My first name is Lhoussaine and my last name is daoudi. "}
        ]
    }
   

)

In [9]:
print(display(Markdown(response['messages'][-1].content)))

It's nice to meet you, Lhoussaine Daoudi! How can I help you today?

None


In [10]:
response=agent.invoke(
    input={
        'messages':[
            {'role':'user','content':"i'm Lhoussaine what is my last name "}
        ]
    }
)

In [11]:
print(display(Markdown(response['messages'][-1].content)))

I don't know your last name. As an AI, I only have access to the information you provide me during our conversation, and you haven't mentioned your last name yet!

None


NB: vous pouvez specifier model

Configuration et controle de modele

Pour mieux contrôler la configuration du modèle, initialisez directement une instance du modèle à l'aide du package du fournisseur. Dans cet exemple, nous utilisons ChatOpenAl. Consultez la section Modèles de chat pour découvrir les autres classes de modèles de chat disponibles.

In [12]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage

In [13]:
llm = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0.1,
    num_predict=1000,
    client_kwargs={"timeout": 120}
)

In [14]:
agent=create_agent(
   model=llm,
   tools=[],
   system_prompt= "You are a helpful assistant",
   debug=True
    )
response =agent.invoke(input={"messages" :[HumanMessage("What is the healthiest time to eat dinner? ")]
})

[values] {'messages': [HumanMessage(content='What is the healthiest time to eat dinner? ', additional_kwargs={}, response_metadata={}, id='66a67bcf-d528-4ef6-b974-a0164d1485c6')]}
[updates] {'model': {'messages': [AIMessage(content='While there is no single "perfect" time that applies to everyone, most nutritionists and sleep experts agree that the healthiest time to eat dinner is **approximately 2 to 3 hours before you go to bed.**\n\nHere is a detailed breakdown of why this timing matters and how to determine the best window for your specific lifestyle.\n\n### 1. The "3-Hour Rule"\nThe primary goal of timing your dinner is to allow your body enough time to digest food before you lie flat to sleep. Eating too close to bedtime can lead to:\n*   **Acid Reflux/GERD:** Lying down with a full stomach allows stomach acid to move back up into the esophagus, causing heartburn.\n*   **Poor Sleep Quality:** Digestion is an active process that raises your core body temperature. For deep sleep, y

In [15]:
print(display(Markdown(response['messages'][-1].content)))

While there is no single "perfect" time that applies to everyone, most nutritionists and sleep experts agree that the healthiest time to eat dinner is **approximately 2 to 3 hours before you go to bed.**

Here is a detailed breakdown of why this timing matters and how to determine the best window for your specific lifestyle.

### 1. The "3-Hour Rule"
The primary goal of timing your dinner is to allow your body enough time to digest food before you lie flat to sleep. Eating too close to bedtime can lead to:
*   **Acid Reflux/GERD:** Lying down with a full stomach allows stomach acid to move back up into the esophagus, causing heartburn.
*   **Poor Sleep Quality:** Digestion is an active process that raises your core body temperature. For deep sleep, your body needs its core temperature to drop.
*   **Blood Sugar Spikes:** Eating a heavy meal right before bed can cause a spike in blood glucose, which may interfere with the release of growth hormones and melatonin.

### 2. Aligning with Your Circadian Rhythm
Your body is biologically programmed to handle nutrients more efficiently during the day than at night. 
*   **Insulin Sensitivity:** Insulin sensitivity generally decreases as the evening progresses. This means your body struggles more to process carbohydrates and sugars late at night, making them more likely to be stored as fat.
*   **Metabolic Slowdown:** While you still burn calories while sleeping, your metabolic rate slows down. Eating a massive meal at 10:00 PM often provides more energy than your body needs for the rest of the night.

### 3. How to Calculate Your Ideal Time
To find your personal "healthy window," work backward from your wake-up time:
*   **If you go to bed at 10:00 PM:** Aim to finish dinner between **6:00 PM and 8:00 PM**.
*   **If you go to bed at midnight:** Aim to finish dinner between **8:00 PM and 10:00 PM**.

### 4. What if you are hungry right before bed?
If you have a long gap between dinner and sleep and find yourself unable to fall asleep due to hunger, avoid a full meal. Instead, opt for a **small, light snack** (under 200 calories) that combines a protein and a complex carb.

**Healthy late-night options:**
*   A small bowl of Greek yogurt with berries.
*   A piece of whole-grain toast with a slice of turkey.
*   A handful of almonds and a piece of fruit.

### Summary Checklist for a Healthy Dinner:
*   **Timing:** 2–3 hours before sleep.
*   **Composition:** Focus on lean proteins and vegetables; keep heavy carbohydrates moderate.
*   **Portion:** Make dinner your lightest or second-lightest meal of the day (the "Eat breakfast like a king, lunch like a prince, and dinner like a pauper" philosophy).

***Disclaimer:** I am an AI, not a doctor. If you have specific medical conditions like diabetes or hypoglycemia, you should consult your physician to determine the best eating schedule for your blood sugar management.*

None


# Implémentation de l'agent via LangChain : Dynamic Model
Les modèles dynamiques sont sélectionnés à l'exécution en fonction de l'état et du contexte actuels. Cela permet une logique de routage avancée et une optimisation des coûts. Pour utiliser un modèle dynamique, créez un middleware à l'aid: du décorateur ` @wrap_model_call ` qui modifie le modèle dans la requête :

In [16]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

In [17]:
basic_model = ChatOllama(model="gemma4:31b-cloud")
advanced_model = ChatOllama(model="gemma4:31b-cloud")
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])
    env=request.runtime.context.get('env', 'test')
    print (env)
    if env=='prod':
       # Use an advanced model for longer conversations
       model = advanced_model
       print ("advanced model selected ....")
    else:
        model = basic_model
        print ("basic model selected ....")
        
    return handler(request.override(model=model))

In [18]:
agent = create_agent(
    model=basic_model, # Default model
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True
)

In [19]:
agent.invoke( input={'messages':[
  HumanMessage('Test 1')]},
  context={'env': 'test'})

[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='c8c198f6-4eb2-4fb1-aea1-35638b2dad3f')]}
test
basic model selected ....
[updates] {'model': {'messages': [AIMessage(content='Test 1 received. I am working correctly. How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T16:25:54.825376025Z', 'done': True, 'done_reason': 'stop', 'total_duration': 258720457, 'load_duration': None, 'prompt_eval_count': 16, 'prompt_eval_duration': None, 'eval_count': 18, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a1887b-75ed-7371-a721-9ce151461dec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 18, 'total_tokens': 34})]}}
[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='c8c198f6-4eb2-4fb1-aea1-35638b2dad3f'), AIMessage(content=

{'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='c8c198f6-4eb2-4fb1-aea1-35638b2dad3f'),
  AIMessage(content='Test 1 received. I am working correctly. How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T16:25:54.825376025Z', 'done': True, 'done_reason': 'stop', 'total_duration': 258720457, 'load_duration': None, 'prompt_eval_count': 16, 'prompt_eval_duration': None, 'eval_count': 18, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a1887b-75ed-7371-a721-9ce151461dec-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 18, 'total_tokens': 34})]}

In [20]:
 agent.invoke(input={'messages':[
  HumanMessage('Test 1')]},
  context={'env':'prod'})

[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='2f2a4fea-1276-4e3a-a3dc-440f263ff099')]}
prod
advanced model selected ....
[updates] {'model': {'messages': [AIMessage(content='Test received! I am working correctly. How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T16:25:59.604583744Z', 'done': True, 'done_reason': 'stop', 'total_duration': 254209513, 'load_duration': None, 'prompt_eval_count': 16, 'prompt_eval_duration': None, 'eval_count': 16, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a1887b-87cd-7407-b1c2-559a90542994-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 16, 'total_tokens': 32})]}}
[values] {'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='2f2a4fea-1276-4e3a-a3dc-440f263ff099'), AIMessage(content

{'messages': [HumanMessage(content='Test 1', additional_kwargs={}, response_metadata={}, id='2f2a4fea-1276-4e3a-a3dc-440f263ff099'),
  AIMessage(content='Test received! I am working correctly. How can I help you today?', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T16:25:59.604583744Z', 'done': True, 'done_reason': 'stop', 'total_duration': 254209513, 'load_duration': None, 'prompt_eval_count': 16, 'prompt_eval_duration': None, 'eval_count': 16, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a1887b-87cd-7407-b1c2-559a90542994-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 16, 'total_tokens': 32})]}

# Agent Memory

In [21]:

from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langgraph.checkpoint.memory import InMemorySaver

In [22]:
agent=create_agent(model=llm,
  system_prompt="You are a helpful assistant",
  checkpointer=InMemorySaver()
)

In [23]:
resp=agent.invoke(
    input={"messages":[HumanMessage("My Name is Daoudi")]},
    config={"configurable":{"thread_id":1}}

   )

print(resp['messages'][-1].content)

Nice to meet you, Daoudi! How can I help you today?


In [24]:
resp=agent.invoke(
   input={ 'messages': [HumanMessage('What is my name')]},
   config={'configurable': {'thread_id': 1}}
)
print(resp['messages'][-1].content)

Your name is Daoudi.


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0.1
)

print("Modèle llm initialisé !")

Modèle llm initialisé !


In [6]:
agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
)

print("Agent créé avec succès !")

Agent créé avec succès !


In [7]:
from getpass import getpass
from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import create_agent

password = getpass("Mot de passe PostgreSQL : ")

DB_URI = (
    f"postgresql://postgres:{password}"
    "@localhost:5432/postgres?sslmode=disable"
)

# Ouvrir la connexion PostgreSQL
checkpointer_cm = PostgresSaver.from_conn_string(DB_URI)
checkpointer = checkpointer_cm.__enter__()

checkpointer.setup()

agent = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer,
)

print("Agent créé avec succès !")

Agent créé avec succès !


In [8]:
from langchain_core.messages import HumanMessage

In [9]:
from langchain_core.messages import HumanMessage

resp = agent.invoke(
    {"messages": [HumanMessage(content="My name is Daoudi")]},
    config={"configurable": {"thread_id": "1"}}
)

print(resp["messages"][-1].content)

Nice to meet you, Daoudi! How can I help you today?


# Agent tools

### Le rôle des outils pour les agents

Les outils donnent aux agents la capacité d'agir. Les agents vont au-delà du simple rattachement d'outils au modèle en facilitant :

* **Plusieurs appels d'outils en séquence :** Déclenchés par un seul prompt.
* **Des appels d'outils en parallèle :** Lorsque cela est pertinent.
* **La sélection dynamique des outils :** En fonction des résultats précédents.
* **La logique de nouvelle tentative (*retry*) :** Ainsi que la gestion des erreurs.
* **La persistance de l'état :** Conservation des données entre les appels d'outils.


In [10]:
from langchain.tools import tool
from langchain.agents import create_agent

In [11]:
@tool
def search(query: str) -> str:
    """ Search for news."""
    print(f'Search tool is called for {query}')
    return{
        'query':query,
        'news':[
            " Le temps est tres glacial",
            "les conditions meteo sont tres delicates"
        ]         
    }
@tool
def get_weather(location: str) -> str:
    """ Get weather information for a location."""
    print(f'Weather tool is called for {location}')
    return "Weather in {location): Sunny, 32°C"

@tool
def get_employee_info(name: str) -> str:
    """Get information aboud the given employee name"""
    print(f'get_employee_info tool is called for {name}')
    return {'name' : name, 'salary': 12000, 'job':'Developper'}

In [12]:
agent=create_agent(
    model=llm,
    tools=[search, get_weather, get_employee_info]
)

In [14]:
from IPython.display import display, Markdown

In [15]:
response=agent.invoke(input={'messages':[HumanMessage('What is the weather in BeniMellal' )]})
print(display(Markdown(response['messages'][-1].content) ))

Weather tool is called for BeniMellal


The weather in Beni Mellal is currently sunny with a temperature of 32°C.

None


In [16]:
response=agent.invoke(input={'messages':[
    HumanMessage('Quel est le job de Lhoussaine et le domaine de recherche' )
    ]
    }
    )
print(display(Markdown(response['messages'][-1].content) ))

get_employee_info tool is called for Lhoussaine


Lhoussaine est **Développeur**. 

Cependant, les informations disponibles ne précisent pas son domaine de recherche.

None


In [17]:
response=agent.invoke(input={'messages':[HumanMessage('What are news' )]})
print(display(Markdown(response['messages'][-1].content) ))

Search tool is called for latest top news headlines today


It seems the current top news highlights are focused on severe weather conditions, specifically reporting that the weather is extremely freezing and conditions are very delicate. 

Since "news" is a very broad topic, could you let me know if you are looking for something specific? For example:
*   **World News** (Politics, International Relations)
*   **Technology** (AI, Gadgets)
*   **Sports**
*   **Entertainment**
*   **Finance/Business**

None


In [18]:
response=agent.invoke(input={'messages':[HumanMessage('What are news' )]})
print(display(Markdown(response['messages'][-1].content) ))

Search tool is called for latest top news headlines today


It seems the current news feed is reporting on severe weather conditions, specifically noting that the weather is freezing and conditions are very delicate.

Since "news" is a very broad topic, could you let me know if you are looking for something specific? For example:
*   **World News** (Politics, International Relations)
*   **Technology** (AI, Gadgets)
*   **Business/Finance** (Stock market, Economy)
*   **Entertainment** (Movies, Music, Celebrities)
*   **Sports**

Let me know what you're interested in, and I'll find the latest updates for you!

None


In [19]:
from ddgs import DDGS

@tool
def web_search(query: str, num_results:int=5) -> str:
    """ 
    Search the web usin DuckDuckGo
    Args:
        query: Search query string
        num_results : Number of results to return (Default=5)
    Returns:
        Formatted search results with titles, descptions and Urls
        """
    
    try:
        print(f'Search tool is called for {query}')
        ddgs_search = DDGS()
        results=ddgs_search.text(query=query, max_results=num_results, backend="google")
        if not results:
            return f"No results found for {query} "
        formatted_results = [f"Search for {query} : \n"]
        for i, result in enumerate(results, 1) :
            title = result.get("title", "No Title")
            body = result.get ("body", "No description available")
            href = result.get("href","")
            formatted_results.append( f"{i}. **(title]**. {body}. {href}")
            return formatted_results
    except Exception as e:
        print(str(e))


In [20]:
llm = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0.1,
    num_predict=1000,
    client_kwargs={"timeout": 120}
)

In [21]:
agent=create_agent(model=llm, tools=[web_search, get_employee_info, get_weather], debug=True)

resp=agent.invoke(input={'messages': [HumanMessage('Search for Agentic RAG tutorials')]})
print(display(Markdown(resp['messages'][-1].content)))

[values] {'messages': [HumanMessage(content='Search for Agentic RAG tutorials', additional_kwargs={}, response_metadata={}, id='8dd5bbe8-7c1f-43db-9711-5606e341548b')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T17:19:12.687628324Z', 'done': True, 'done_reason': 'stop', 'total_duration': 441075394, 'load_duration': None, 'prompt_eval_count': 213, 'prompt_eval_duration': None, 'eval_count': 19, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a188ac-410f-7677-bef3-8288481d9e06-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'Agentic RAG tutorials'}, 'id': '9ce19da4-3a55-4bb7-a4ef-05aeb178cdb5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 213, 'output_tokens': 19, 'total_tokens': 232})]}}
[values] {'messages': [HumanMessage(content='Search for Agentic RAG tutorials', additional_kwar

I found a helpful tutorial to get you started with Agentic RAG:

**[Agentic RAG: Step-by-Step Tutorial With Demo Project](https://www.datacamp.com/tutorial/agentic-rag-tutorial)**
This guide from DataCamp teaches you how to build an Agentic RAG pipeline from scratch. It covers integrating local data sources and web scraping to ensure the AI generates context-aware responses to user queries.

### What is Agentic RAG? (Quick Overview)
Unlike standard RAG (Retrieval-Augmented Generation), which follows a linear path (Retrieve $\rightarrow$ Augment $\rightarrow$ Generate), **Agentic RAG** uses an AI agent to manage the process. This allows the system to:
*   **Iterate:** If the first retrieval doesn't answer the question, the agent can try a different search query.
*   **Reason:** The agent can decide which tool to use (e.g., a vector database for internal docs vs. a web search for current events).
*   **Self-Correct:** The agent can evaluate the retrieved documents for relevance and discard irrelevant information before generating the final answer.

None


In [22]:
from langchain.agents import create_agent 
from langchain.chat_models import init_chat_model
from langchain.tools import tool
import asyncio
from datetime import datetime
from langchain_tavily import TavilySearch
tavily_search_tool = TavilySearch(max_results=10, search_depth="advanced")

@tool
def search(query: str) -> str:
    """
    Search for general web results
    """
    print("Search Tool invoked")
    return tavily_search_tool.invoke({"query": query})

In [24]:
from dotenv import load_dotenv

In [25]:
load_dotenv(override=True)

True

In [26]:
model=init_chat_model(model="gemma4:31b-cloud", model_provider="ollama", temperature=0)

agent=create_agent(
    model=model,
    tools=[search],
    system_prompt=f"""You are a helpful assistant that serach
                   the web for information using search tool
                  today's date is {datetime.now().strftime("%Y-%m-%d") }
                  """,
)

In [27]:
resp=agent.invoke(input={'messages': [HumanMessage("Quel temps fait-il aujourd'hui à BeniMellal")]})
print(display(Markdown( resp['messages'][-1].content) ))

Search Tool invoked


Aujourd'hui, le jeudi 29 octobre 2026, le temps à **Beni Mellal** est globalement agréable avec un ciel variable (nuages épars).

Voici les détails pour la journée :
*   **Températures :** Les maximales atteignent environ **27°C à 28°C** l'après-midi, et les minimales descendent autour de **16°C à 17°C**.
*   **Ciel :** Alternance de nuages et d'éclaircies.
*   **Vent :** Vent modéré soufflant principalement du nord-ouest (environ 10 à 13 km/h).
*   **Précipitations :** Aucune pluie significative n'est prévue pour aujourd'hui.
*   **Qualité de l'air :** Elle est jugée bonne, idéale pour les activités en plein air.

**À noter pour demain (vendredi 30 octobre) :** Le temps devrait changer avec des prévisions de pluies faibles à modérées et un ciel variable, tout en conservant des températures similaires (max 28-29°C).

None


# PythonREPL Tool

In [28]:
from langchain_experimental.utilities import PythonREPL
python_repl = PythonREPL()
python_repl.run( 'print(f"la somme de 5 et 6 est {5+9}") ')

C:\Users\jihan\AppData\Local\Temp\ipykernel_21312\2779026968.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL
Python REPL can execute arbitrary code. Use with caution.


'la somme de 5 et 6 est 14\n'

In [29]:
from langchain_core.tools import Tool
from langchain.tools import tool, ToolRuntime
repl_tool=Tool(
    name="repl_tool",
    description=" A Python shell used to execute python commands. Input should be a valid python command",
    func= python_repl.run
)


In [30]:
repl_tool.invoke("""
a=5
b=9
print(f"la somme de {a} et {b} est {a+b}")
""")

'la somme de 5 et 9 est 14\n'

In [31]:
from langchain.chat_models import init_chat_model
llm=init_chat_model("gemma4:31b-cloud", model_provider="ollama", temperature=0)

In [32]:
agent=create_agent(
    model=llm, tools=[repl_tool],
    debug=True,
    system_prompt='generate python code and use the repl tool to execute'
)

In [33]:
resp= agent.invoke(input={'messages':[HumanMessage(
"""
a= 5
b=9
print(f"la somme de {a} et {b} est {a+b}")
"""
)]})
print(display(Markdown(resp['messages'][-1].content)))

[values] {'messages': [HumanMessage(content='\na= 5\nb=9\nprint(f"la somme de {a} et {b} est {a+b}")\n', additional_kwargs={}, response_metadata={}, id='d8e71abc-2905-49eb-91f6-7651f9a4da40')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T17:23:01.574187699Z', 'done': True, 'done_reason': 'stop', 'total_duration': 484589616, 'load_duration': None, 'prompt_eval_count': 114, 'prompt_eval_duration': None, 'eval_count': 46, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a188af-bee9-72cc-be9f-1e9735eb5f9a-0', tool_calls=[{'name': 'repl_tool', 'args': {'__arg1': 'a = 5\nb = 9\nprint(f"la somme de {a} et {b} est {a+b}")'}, 'id': '87ed1873-fa81-4b5c-bfd1-b0940dbc1ae2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 46, 'total_tokens': 160})]}}
[values] {'messages': [HumanMessage

la somme de 5 et 9 est 14

None


In [34]:
resp=agent.invoke(input={'messages':[HumanMessage(
""" 
Je veux trier ces deux listes : liste1=[5,3,8], liste2=(1,9,3] ensuite additionner les deux listes.
"""
)]})
print(display(Markdown(resp['messages'][-1].content)))

[values] {'messages': [HumanMessage(content=' \nJe veux trier ces deux listes : liste1=[5,3,8], liste2=(1,9,3] ensuite additionner les deux listes.\n', additional_kwargs={}, response_metadata={}, id='a1cddd14-44df-4f86-85e9-8b5a29c12e0b')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T17:23:15.420515704Z', 'done': True, 'done_reason': 'stop', 'total_duration': 878316056, 'load_duration': None, 'prompt_eval_count': 120, 'prompt_eval_duration': None, 'eval_count': 165, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a188af-f248-738e-b5e2-5adc05ad9cb6-0', tool_calls=[{'name': 'repl_tool', 'args': {'__arg1': 'liste1 = [5, 3, 8]\nliste2 = [1, 9, 3]\n\n# Trier les listes\nliste1_triee = sorted(liste1)\nliste2_triee = sorted(liste2)\n\n# Additionner les éléments des deux listes (élément par élément)\nresultat = [x + y for x, y in 

Voici le code Python pour réaliser ces opérations. 

*Note : J'ai corrigé la syntaxe de votre `liste2` car vous aviez utilisé une parenthèse et un crochet `(1,9,3]`, ce qui cause une erreur en Python.*

```python
liste1 = [5, 3, 8]
liste2 = [1, 9, 3]

# 1. Trier les listes
liste1_triee = sorted(liste1)
liste2_triee = sorted(liste2)

# 2. Additionner les deux listes (élément par élément)
# On utilise zip() pour coupler les éléments des deux listes triées
resultat = [x + y for x, y in zip(liste1_triee, liste2_triee)]

print(f"Liste 1 triée : {liste1_triee}")
print(f"Liste 2 triée : {liste2_triee}")
print(f"Résultat de l'addition : {resultat}")
```

**Résultat obtenu :**
*   Liste 1 triée : `[3, 5, 8]`
*   Liste 2 triée : `[1, 3, 9]`
*   Résultat de l'addition : `[4, 8, 17]` (car 3+1=4, 5+3=8, 8+9=17)

None


In [35]:
from langchain_experimental.tools import PythonREPLTool
from langchain.messages import SystemMessage, HumanMessage

In [36]:
agent4 = create_agent(
    model=llm,
    tools=[PythonREPLTool(sanitize_input=False)],
    system_prompt=SystemMessage("""
             Generate the python code Use the REPL Tool
             to execute the generated code Write
             the generated python code and the execution
             result in a file CodeReslt.txt 
             """),
    debug=True
)

In [37]:
resp = agent4.invoke(input={
'messages':[
HumanMessage(content="""Utilise sympy ou numpy pour trouver les racines exactes du polynôme $6x^2 + 11x - 6 = 0$.""")]
})

[values] {'messages': [HumanMessage(content='Utilise sympy ou numpy pour trouver les racines exactes du polynôme $6x^2 + 11x - 6 = 0$.', additional_kwargs={}, response_metadata={}, id='e1a6ae44-f303-45f9-98c7-fa62430a0ea4')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-29T17:23:34.518126572Z', 'done': True, 'done_reason': 'stop', 'total_duration': 952594003, 'load_duration': None, 'prompt_eval_count': 164, 'prompt_eval_duration': None, 'eval_count': 91, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a188b0-3c8e-76c4-bb9b-3f1af7963627-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'import sympy as sp\n\n# Define the variable\nx = sp.symbols(\'x\')\n\n# Define the polynomial equation\nequation = 6*x**2 + 11*x - 6\n\n# Solve for x\nroots = sp.solve(equation, x)\n\nprint(f"Les racines exactes du polynôme sont : {roots

In [38]:
print(resp['messages'][-1].content)

Voici le code Python pour trouver les racines du polynôme $6x^2 + 11x - 6 = 0$. 

Puisque `sympy` n'était pas disponible dans l'environnement d'exécution immédiat, j'ai utilisé `numpy` pour une approximation numérique et j'ai implémenté la formule quadratique pour obtenir la forme exacte.

```python
import numpy as np
import math

# Coefficients du polynôme 6x^2 + 11x - 6 = 0
a = 6
b = 11
c = -6

# 1. Utilisation de numpy pour les racines numériques
coeffs = [a, b, c]
roots_np = np.roots(coeffs)

# 2. Calcul manuel pour les racines exactes (Formule quadratique)
# Delta = b^2 - 4ac
delta = b**2 - 4*a*c
sqrt_delta = math.sqrt(delta)

x1_exact = f"(-{b} + sqrt({delta})) / {2*a}"
x2_exact = f"(-{b} - sqrt({delta})) / {2*a}"

x1_val = (-b + sqrt_delta) / (2*a)
x2_val = (-b - sqrt_delta) / (2*a)

print(f"Racines numériques (numpy) : {roots_np}")
print(f"Delta : {delta}")
print(f"Racines exactes : x1 = {x1_exact}, x2 = {x2_exact}")
print(f"Valeurs approx : x1 = {x1_val}, x2 = {x2_val}")
```



# Tool handlyng errr 

In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
@wrap_tool_call
def handle_tool_errors(request, handler):
   """ Handl tool execution errors with custom messages."""
   try:
      return handler(request)
   except Exception as e:
     print ('Errorrrr')
     # Return a custom error message to the model
     return ToolMessage(
        content=f"Tool error: Please check your input and try again. ({str(e)})",
        tool_call_id=request.tool_call["id"]
    )
agent = create_agent(
    model=llm,
    tools=[search, get_weather],
    middleware=[handle_tool_errors], debug=True
)

# controle de System Prompt

In [40]:
from typing import TypedDict
from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from IPython.display import Markdown

class Context(TypedDict):
    user_role: str

@dynamic_prompt 
def user_role_prompt(request: ModelRequest) -> str:
    """ Generatê system prompt based on user role."""
    user_role = request.runtime.context.get ("user_role", "user")
    base_prompt = "You are a helpful assistant."
    print (user_role)
    if user_role == "expert":
       system_prompt= f"{base_prompt} Provide detailed technical responses."
       print(f"Model with System Prompt : {system_prompt}")
       return system_prompt
    elif user_role == "beginner":
         system_prompt= f"(base_prompt) Explain concepts simply and avoid jargon."
         print (f"Model with System Prompt : {system_prompt}")
         return system_prompt
    
    return base_prompt


In [41]:
agent =create_agent(
    model=llm, tools=[],
    middleware= [user_role_prompt],
    context_schema=Context,
    debug=True
)

In [42]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain AI agent"}]},
    context={"user_role": "expert"})
print(display(Markdown(result['messages'][-1].content) ))

[values] {'messages': [HumanMessage(content='Explain AI agent', additional_kwargs={}, response_metadata={}, id='8ff7c6e1-0b7e-4a38-b37c-b9071a6065de')]}
expert
Model with System Prompt : You are a helpful assistant. Provide detailed technical responses.
[updates] {'model': {'messages': [AIMessage(content='At its simplest level, an **AI Agent** is a system that uses a Large Language Model (LLM) as its "brain" to not only generate text but to **perceive its environment, reason about how to achieve a goal, and take actions to accomplish that goal.**\n\nWhile a standard chatbot (like a basic LLM) is **reactive** (it answers a prompt and stops), an AI Agent is **proactive** (it plans a sequence of steps and executes them until the task is complete).\n\n---\n\n### 1. The Core Architecture of an AI Agent\nTo understand how an agent works, think of it as a loop consisting of four primary components:\n\n#### A. The Brain (The LLM)\nThe LLM provides the reasoning capabilities. It handles:\n*   *

At its simplest level, an **AI Agent** is a system that uses a Large Language Model (LLM) as its "brain" to not only generate text but to **perceive its environment, reason about how to achieve a goal, and take actions to accomplish that goal.**

While a standard chatbot (like a basic LLM) is **reactive** (it answers a prompt and stops), an AI Agent is **proactive** (it plans a sequence of steps and executes them until the task is complete).

---

### 1. The Core Architecture of an AI Agent
To understand how an agent works, think of it as a loop consisting of four primary components:

#### A. The Brain (The LLM)
The LLM provides the reasoning capabilities. It handles:
*   **Planning:** Breaking a complex goal (e.g., "Research this company and write a summary") into smaller steps.
*   **Decision Making:** Deciding which tool to use or whether the current result is satisfactory.
*   **Self-Correction:** Reviewing its own output and fixing errors.

#### B. Planning (The Strategy)
Agents don't just guess; they use specific frameworks to think:
*   **Chain-of-Thought (CoT):** The agent prompts itself to "think step-by-step."
*   **Reflection/Self-Criticism:** The agent looks at its own draft and asks, "Does this actually answer the user's request?" and iterates.
*   **Task Decomposition:** Breaking a large goal into a "To-Do" list.

#### C. Memory (The Context)
Agents need to remember what they have already tried so they don't loop infinitely.
*   **Short-term Memory:** The immediate conversation history (Context Window).
*   **Long-term Memory:** Usually implemented via a **Vector Database** (RAG - Retrieval Augmented Generation), allowing the agent to store and retrieve documents or past experiences.

#### D. Tools/Action Space (The Hands)
This is what separates an agent from a chatbot. An agent is given a set of **APIs or functions** it is allowed to call. Examples include:
*   **Web Browser:** To search for real-time information.
*   **Code Interpreter:** To write and run Python code for math or data analysis.
*   **Database Access:** To query a company's SQL database.
*   **App Integration:** To send an email via Gmail or create a ticket in Jira.

---

### 2. The Agentic Workflow (The Loop)
An AI agent typically operates in a loop known as **ReAct (Reason + Act)**:

1.  **Input:** User says, "Find the current price of Nvidia stock and email it to my boss."
2.  **Thought:** "I need to find the current stock price of NVDA. I should use the Google Search tool."
3.  **Action:** Calls `google_search("NVDA stock price")`.
4.  **Observation:** Receives the result: "$120.45".
5.  **Thought:** "I have the price. Now I need to send an email. I should use the Gmail tool."
6.  **Action:** Calls `send_email(to="boss@company.com", body="NVDA is $120.45")`.
7.  **Final Response:** "I have found the price and emailed your boss."

---

### 3. Types of AI Agents
Depending on the complexity, agents are categorized as:

*   **Single-Agent Systems:** One LLM handling everything (e.g., a personal assistant).
*   **Multi-Agent Systems (MAS):** Multiple specialized agents working together. For example:
    *   **Agent A (Researcher):** Finds the data.
    *   **Agent B (Writer):** Turns data into a report.
    *   **Agent C (Editor):** Checks the report for facts and grammar.
    *   *This reduces "hallucinations" because agents check each other's work.*

---

### 4. Comparison: Chatbot vs. AI Agent

| Feature | Standard Chatbot | AI Agent |
| :--- | :--- | :--- |
| **Behavior** | Reactive (Question $\rightarrow$ Answer) | Proactive (Goal $\rightarrow$ Plan $\rightarrow$ Action) |
| **Capability** | Text generation/summarization | Task execution/Tool usage |
| **Autonomy** | Low (Requires prompt for every step) | High (Can run autonomously for hours) |
| **State** | Stateless (mostly) | Stateful (maintains memory of progress) |
| **Example** | "Write a poem about cats." | "Book a flight to Tokyo under $800." |

### 5. Technical Challenges
Despite their power, agents face several hurdles:
*   **Infinite Loops:** The agent might get stuck repeating the same failed action.
*   **Error Propagation:** If the agent makes a mistake in Step 1, the rest of the plan fails.
*   **Security (Prompt Injection):** If an agent has access to your email, a malicious website it reads could "trick" the agent into emailing your passwords to a hacker.
*   **Latency:** Reasoning and tool-calling take significantly longer than simple text generation.

None


In [43]:
result = agent.invoke(
{"messages": [{"role": "user", "content": "Explain AI Agent"}]},
context={"user_role": "beginner"})
print(display(Markdown(result['messages'][-1].content) ))

[values] {'messages': [HumanMessage(content='Explain AI Agent', additional_kwargs={}, response_metadata={}, id='4d52ba48-ed29-48bc-bb43-45def55b1c26')]}
beginner
Model with System Prompt : (base_prompt) Explain concepts simply and avoid jargon.
[updates] {'model': {'messages': [AIMessage(content='Think of a standard AI (like a basic chatbot) as a **smart book**. You ask it a question, and it gives you an answer based on what it knows. It doesn\'t *do* anything; it just talks.\n\nAn **AI Agent**, however, is like a **smart employee**. It doesn\'t just give you information; it uses that information to complete a goal.\n\nHere is the simplest way to understand the difference:\n\n### The Basic AI vs. The AI Agent\n\n*   **Basic AI:** You ask, "What is the weather in Tokyo?" $\\rightarrow$ It tells you, "It is 65 degrees and rainy."\n*   **AI Agent:** You tell it, "I\'m going to Tokyo next week. Find a hotel under $200, book it, and add the reservation to my calendar." $\\rightarrow$ The ag

Think of a standard AI (like a basic chatbot) as a **smart book**. You ask it a question, and it gives you an answer based on what it knows. It doesn't *do* anything; it just talks.

An **AI Agent**, however, is like a **smart employee**. It doesn't just give you information; it uses that information to complete a goal.

Here is the simplest way to understand the difference:

### The Basic AI vs. The AI Agent

*   **Basic AI:** You ask, "What is the weather in Tokyo?" $\rightarrow$ It tells you, "It is 65 degrees and rainy."
*   **AI Agent:** You tell it, "I'm going to Tokyo next week. Find a hotel under $200, book it, and add the reservation to my calendar." $\rightarrow$ The agent searches the web, compares prices, uses your credit card (with permission), and updates your calendar.

---

### How does an AI Agent actually work?

To act like an "employee," an agent follows a simple loop:

1.  **The Goal:** You give it a destination (e.g., "Plan a trip").
2.  **The Plan:** The agent thinks, *"First I need to find flights, then a hotel, then a car."* It breaks the big goal into small steps.
3.  **The Tools:** The agent has a "toolbox." This could be access to Google Search, your email, a calculator, or a travel booking website.
4.  **The Action:** It uses a tool to do a task.
5.  **The Review:** It looks at the result. If the hotel it found is too expensive, it says, *"That didn't work,"* and tries a different search. It keeps going until the job is done.

### A Real-World Example
Imagine an **AI Customer Service Agent**. 

Instead of just telling a customer how to return a package (Basic AI), the **Agent** can:
*   Look up the customer's order number in the database.
*   Check if the item is still eligible for a return.
*   Email a shipping label to the customer.
*   Notify the warehouse that a return is coming.

### Summary
*   **AI** = Knowledge and conversation.
*   **AI Agent** = Knowledge + Planning + Action.

None


# Structured output

In [44]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ContactInfo(BaseModel) :
    name: str 
    email: str 
    phone: str

agent= create_agent(
    model=llm,
    tools=[], 
    response_format=ToolStrategy(ContactInfo)
    # response_format=ProviderStrategy(ContactInfo)
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: Lhoussaine DAOUDI,  lhoussaine.daoudi@usms.ma, (212)254430044"}],
})

contact=result["structured_response"]
#ContactInfo(name='lhoussaine', email=.....)    

In [45]:
print(contact.name)
print(contact.email)
print(contact.phone)

Lhoussaine DAOUDI
lhoussaine.daoudi@usms.ma
(212)254430044


# Custom Middleware

In [46]:
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.messages import HumanMessage
import time

class HooksDemo(AgentMiddleware):
    def __init__(self) :
     super().__init__()
     self.starttime = 0.0
    def before_agent(self, state: AgentState, runtime):
        self.starttime = time.time()
        print('Befor_agent triggered')
    def before_model(self, state : AgentState, runtime):
        print ('Beforre_model triggered')
    def after_model (self, state: AgentState, runtime):
        print('After Model triggered')
    def after_agent(self, state : AgentState, runtime):
        print( 'After Agent triggered')
        print("duration =", time. time()-self.starttime)

In [47]:
agent=create_agent(model=llm, middleware=[HooksDemo()])
res=agent.invoke({'messages': [HumanMessage('Tel me how many cities in morocco' )]})

Befor_agent triggered
Beforre_model triggered
After Model triggered
After Agent triggered
duration = 1.954441785812378


# RAG As Agent Tool

In [48]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model=OllamaEmbeddings(model="nomic-embed-text")

C:\Users\jihan\AppData\Local\Temp\ipykernel_21312\2501881381.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [49]:
texts = [
    "Je m'appelle Lhoussaine Daoudi. Je suis professeur en informatique.",
    "Je travaille à la FST de Béni Mellal, USMS.",
    "J'ai obtenu mon doctorat d'État en 2025.",
    "En plus de l'informatique, je suis passionné par la musique et la culture.",
    "Je me passionne pour la philosophie.",
    "Je suis originaire de Zagora, une ville du sud-est du Maroc.",
    "J'ai fait mes études au lycée Salah Eddine Al Ayoubi à Tinghir.",
    "Après le baccalauréat, j'ai étudié à la FST d'Errachidia pendant 4 ans.",
]


In [50]:
vectore_store= FAISS.from_texts(texts=texts, embedding=embedding_model)
results=vectore_store.similarity_search("Nom, prénom et affiliation", 2)

In [51]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage
@wrap_tool_call
def handle_tool_errors(request, handler):
   """ Handl tool execution errors with custom messages."""
   try:
      return handler(request)
   except Exception as e:
     print ('Errorrrr')
     # Return a custom error message to the model
     return ToolMessage(
        content=f"Tool error: Please check your input and try again. ({str(e)})",
        tool_call_id=request.tool_call["id"]
    )
agent = create_agent(
    model=llm,
    tools=[search, get_weather],
    middleware=[handle_tool_errors], debug=True
)

In [52]:
print(results[0].page_content)
print(results[1].page_content)


Je travaille à la FST de Béni Mellal, USMS.
Je me passionne pour la philosophie.


In [53]:
from langchain_core.tools import create_retriever_tool

retrieval=vectore_store.as_retriever(kwargs={'k':5})
retrieval_tool=create_retriever_tool(
    retriever=retrieval,
    name="kb_search",
    description="Search information about me"
)

In [54]:
search_agent =create_agent(
    model=llm,
    system_prompt="Search infomration about me",
    tools= [retrieval_tool]
)

In [55]:
result=search_agent.invoke({'messages':[HumanMessage("Nom et Prénom, Affiliation et diplômes")]})
print(display(Markdown(result['messages'][-1].content)))

Voici les informations demandées :

*   **Nom et Prénom :** Lhoussaine Daoudi
*   **Affiliation :** Professeur en informatique à la FST de Béni Mellal, Université Sultan Moulay Slimane (USMS)
*   **Diplôme :** Doctorat d'État (obtenu en 2025)

None


# Guardrails
Les garde-fous vous aident à concevoir des applications d'IA sûres et conformes en validant et en filtrant le contenu à des étapes clés de l'exécution de votre agent. Ils peuvent détecter des informations sensibles, appliquer des politiques de contenu, valider les sorties et prévenir les comportements dangereux avant qu'ils ne causent des problèmes.

**Les cas d'usage courants incluent :**

* Empêcher la fuite de données personnelles identifiables (PII)
* Détecter et bloquer les attaques par injection de prompt
* Bloquer les contenus inappropriés ou nuisibles
* Appliquer les règles métier et les exigences de conformité
* Valider la qualité et l'exactitude des résultats

Vous pouvez mettre en œuvre des garde-fous à l'aide d'un middleware pour intercepter l'exécution à des points stratégiques — avant le démarrage de l'agent, après son exécution, ou autour des appels au modèle et aux outils.

Les garde-fous peuvent être mis en œuvre selon deux approches complémentaires :

* **Garde-fous déterministes :** Ils utilisent une logique fondée sur des règles, comme des expressions régulières (regex), la correspondance de mots-clés ou des vérifications explicites. Ils sont rapides, prévisibles et économiques, mais peuvent manquer certaines violations plus subtiles.
* **Garde-fous fondés sur des modèles :** Ils utilisent des LLM ou des classificateurs pour évaluer le contenu avec une compréhension sémantique. Ils permettent de détecter des problèmes subtils que les règles seules ne repèrent pas, mais sont plus lents et plus coûteux.

LangChain fournit à la fois des garde-fous intégrés (par exemple, la détection de PII et l'intervention humaine dans la boucle) ainsi qu'un système de middleware flexible pour créer des garde-fous personnalisés selon l'une ou l'autre approche.

# Garde-fous intégrés

## Détection de PII

LangChain fournit un middleware intégré pour détecter et traiter les informations personnelles identifiables (PII) dans les conversations. Ce middleware peut détecter les types de PII courants, comme les adresses e-mail, les numéros de carte bancaire, les adresses IP, et bien d'autres.

L'image ci-dessous illustre le flux d'exécution et les différents points d'interception (hooks) où ces garde-fous peuvent être appliqués :

![Schéma des points d'interception des garde-fous](image_1.jpeg)

Le middleware de détection des PII est utile dans des cas tels que les applications de santé et de finance soumises à des exigences de conformité, les agents de service client qui doivent assainir les journaux, ainsi que, plus généralement, toute application manipulant des données utilisateur sensibles.

Le middleware PII prend en charge plusieurs stratégies pour traiter les PII détectées :

| Stratégie | Description | Exemple |
| :--- | :--- | :--- |
| `redact` | Remplace par `[REDACTED_{PII_TYPE}]` | `[REDACTED_EMAIL]` |
| `mask` | Masque partiellement l'information (par ex. les 4 derniers chiffres) | `****-****-****-1234` |
| `hash` | Remplace par un hachage déterministe | `a8f5f167...` |
| `block` | Déclenche une exception lorsqu'une PII est détectée | Erreur levée |

### Types de PII intégrés

* `email` — Adresses e-mail
* `credit_card` — Numéros de carte bancaire (validés avec l'algorithme de Luhn)
* `ip` — Adresses IP
* `mac_address` — Adresses MAC
* `url` — URL

# Options de configuration

| Paramètre | Description | Valeur par défaut |
| :--- | :--- | :--- |
| `pii_type` | Type de PII à détecter (intégré ou personnalisé) | Obligatoire |
| `strategy` | Façon de traiter les PII détectées (`"block"`, `"redact"`, `"mask"`, `"hash"`) | `"redact"` |
| `detector` | Fonction de détection personnalisée ou motif regex | `None` (utilise le détecteur intégré) |
| `apply_to_input` | Vérifie les messages utilisateur avant l'appel au modèle | `True` |
| `apply_to_output` | Vérifie les messages de l'IA après l'appel au modèle | `False` |
| `apply_to_tool_results` | Vérifie les messages de résultat des outils après leur exécution | `False` |

In [56]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

@tool
def card_tool(credit_card:str):
    """
    Process The payment using the credit card
    """
    print(f"Process The payment using the credit card:{credit_card}")
    return f"The payment has been processed succefully with {credit_card}"

@tool
def send_email(email_adress:str):
    """
    Sending The Email using the email_address
    """
    print(f"Sending The Email using the email_address: {email_adress} ")
    return f"Email sent succefully to {email_adress}"

@tool
def process_api_key(api_key: str):
    """
    Process with the api key
    """
    print(f"Process with the api key : {api_key}")
    return f"Processing with the api key: {api_key}"

agent = create_agent(
    model=llm,
    tools=[card_tool, send_email],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
           "api_key",
           detector=r"sk-[a-zA—Z0-9]{32}",
           strategy="block",
           apply_to_input=True,
        ),
    ],
)

In [57]:
# When user provides PII, it will be handled according to the strategy
result = agent.invoke({
    "messages": [{"role": "user", "content": "My email is lhoussainedaoudi@gmail.com and card is 45000-7389-4320-7"}]
})

In [58]:
print(result['messages'][-1].content)

I have received your email address and card information. How can I help you with them today?


# Intervention humaine dans la boucle (Human-in-the-loop)

LangChain fournit un middleware intégré permettant d'exiger une approbation humaine avant l'exécution d'opérations sensibles. Il s'agit de l'un des garde-fous les plus efficaces pour les décisions à forts enjeux.

Le middleware d'intervention humaine dans la boucle est utile dans des cas tels que :
* Les transactions et transferts financiers
* La suppression ou la modification de données de production
* L'envoi de communications à des parties externes
* Toute opération ayant un impact significatif sur l'activité

In [59]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def send_email(email_adress:str):
    """
    Sending The Email using the email_address
    """
    print(f"Sending The Email using the email_address: {email_adress}")
    return f"Email sent succefully to {email_adress}"
agent= create_agent(
    model=llm,
    tools=[send_email],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # Require approval for sensitive operations
                "send_email":True,
            }
        ),
    ],
    # Persist the state across interrupts
    checkpointer=InMemorySaver(),
)
# Human-in-the-loop requires a thread ID for persistence
config={"configurable": {"thread_id": "some_id"}}

# Agent will pause and wait for approval before executing sensitive tools
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to lhoussainedaoudi@gmail.com"}]},
    config=config
)

In [60]:
print(result['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'email_adress': 'lhoussainedaoudi@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'email_adress': 'lhoussainedaoudi@gmail.com'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='88a3726866f838ffbc949eb60ca38627', response_schema=None)]


In [61]:
result=agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config # same thread ID to resume the paused conversation
)

Sending The Email using the email_address: lhoussainedaoudi@gmail.com


In [62]:
print(result['messages'][-1].content)

Email sent successfully to lhoussainedaoudi@gmail.com.


# Garde-fous personnalisés

Pour des garde-fous plus sophistiqués, vous pouvez créer un middleware personnalisé qui s'exécute avant ou après l'exécution de l'agent. Cela vous donne un contrôle total sur la logique de validation, le filtrage du contenu et les vérifications de sécurité.

## Garde-fous avant l'agent

Utilisez les hooks « avant l'agent » pour valider les requêtes une seule fois, au début de chaque invocation. Cela est utile pour des vérifications au niveau de la session, comme l'authentification, la limitation du débit (rate limiting) ou le blocage des requêtes inappropriées avant le début de tout traitement.

In [63]:
from typing import Any
from langchain.agents.middleware import before_agent, AgentState, hook_config
from langgraph.runtime import Runtime

banned_keywords=["hack", "exploit", "malware"]


@tool
def search(query: str) -> str:
    """
    Search for general web results
    """
    print("Search Tool invoked")
    return tavily_search_tool.invoke({"query": query})

@before_agent(can_jump_to=["end"])
def content_filter(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Deterministic guardrail: Block requests containing banned keywords."""
    # Get the first user message
    if not state["messages"]:
        return None

    first_message = state["messages"][0]
    if first_message.type != "human":
        return None
    
    content=first_message.content.lower()

    # Check for banned keywords
    for keyword in banned_keywords:
        if keyword in content:
            # Block execution before any processing
            return {
                "messages": [{
                    "role": "assistant",
                    "content": "I cannot process requests containing inappropriate content. Please repeat"
                }],
                "jump_to": "end"
            }
        
    return None        

# Use the custom guardrail
from langchain. agents import create_agent

agent = create_agent(
    model=llm,
    tools= [search],
    middleware=[content_filter],
)

# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a database?"}]
})

In [64]:
print(result['messages'][-1].content)

I cannot process requests containing inappropriate content. Please repeat


## Garde-fous après l'agent

Utilisez les hooks « après l'agent » pour valider les sorties finales une seule fois avant de les renvoyer à l'utilisateur. 

Cela est utile pour les vérifications de sécurité fondées sur un modèle, la validation de la qualité ou les contrôles finaux de conformité sur la réponse complète de l'agent.

In [65]:
from langchain.agents.middleware import after_agent, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.messages import AIMessage
from langchain.chat_models import init_chat_model
from typing import Any

safety_model=init_chat_model("gemma4:31b-cloud", model_provider="ollama")

@after_agent(can_jump_to=["end"])
def safety_guardrail(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Model-based guardrail: Use an LLM to evaluate response safety."""
    # Get the final AI response
    if not state["messages"]:
        return None
    
    last_message = state["messages"][-1]
    if not isinstance(last_message, AIMessage):
        return None
    
# Use a model to evaluate safety
    safety_prompt = f"""Evaluate if this response is safe and appropriate.
    Respond with only 'SAFE' or 'UNSAFE'.
    Response:{last_message.content}"""

    result = safety_model.invoke([{"role": "user", "content": safety_prompt}])

    if "UNSAFE" in result.content:
       last_message.content = "I cannot provide that response. Please rephrase your request."

    return None

# Use the safety guardrail
from langchain. agents import create_agent

agent=create_agent(
    model=llm,
    tools=[search],
    middleware=[safety_guardrail],
)

result=agent.invoke({
    "messages": [{"role": "user", "content": "How do I make explosives?"}]
})

In [66]:
print(result['messages'][-1].content)

I cannot fulfill this request. I am programmed to be a helpful and harmless AI assistant. My safety guidelines prohibit me from providing instructions, recipes, or guidance on how to create explosives, weapons, or any other harmful materials.


In [67]:
from langchain_tavily import TavilySearch
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain. tools import tool

tavily_search_tool=TavilySearch(max_results=10, search_depth="advanced")

@tool
def search_tool(query: str) -> str:
    """
    Search for general web results
    """
    print("Search Tool invoked")
    return tavily_search_tool.invoke({"query": query})

In [68]:
llm = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0.1,
    num_predict=500,
)

In [69]:
resp=llm.bind_tools([search_tool]).invoke(input=[
    HumanMessage("Météo à Marrakech")
])

In [70]:
print(resp.content)
print(resp.tool_calls)


[{'name': 'search_tool', 'args': {'query': "météo Marrakech aujourd'hui et prévisions"}, 'id': 'bb68a599-3714-427e-ad74-b3b48c8ef83c', 'type': 'tool_call'}]


In [71]:
print("test")

test


# LangGraph